In [17]:
import numpy as np
from numpy.random import uniform
import matplotlib.pyplot as plt

def get_distance(r1, r2):
    return np.sqrt(np.sum((r1 - r2)**2))

def get_transform_pair(r1, r2, distance):
    delta = r1 - r2
    cos = delta[0] / distance
    sin = delta[1] / distance
    transform = np.array([
        [cos,  sin],
        [-sin, cos],
    ])
    reverse = np.array([
        [cos, -sin],
        [sin,  cos],
    ])
    return transform, reverse

def collision(r1, r2, v1, v2, distance):
    transform, reverse = get_transform_pair(r1, r2, distance)
    v1_trans = np.dot(transform, v1)
    v2_trans = np.dot(transform, v2)
    v1_trans[0], v2_trans[0] = v2_trans[0], v1_trans[0]
    v1_reverse = np.dot(reverse, v1_trans)
    v2_reverse = np.dot(reverse, v2_trans)
    return v1_reverse, v2_reverse

def one_step(coords, speeds, bounds):
    coords += speeds

    # столкновение с внешними стенками
    mask = np.logical_or(coords<=bounds[0], coords>=bounds[1])
    coords[mask] -= speeds[mask]
    speeds[mask] = -speeds[mask]
    
    # столкновение с внутренней стенкой
    left, right = bounds[1]//2 - 1, bounds[1]//2 + 1
    mask2 = np.logical_and(coords[:, 0]<=right, coords[:, 0]>=left)
    coords[mask2, 0] -= speeds[mask2, 0]
    speeds[mask2, 0] = -speeds[mask2, 0]
    
    # соударение шариков
    for i in range(len(coords)-1):
        for j in range(i+1, len(coords)):
            r1, r2, v1, v2 = coords[i], coords[j], speeds[i], speeds[j]
            distance = get_distance(r1, r2)
            if distance <= d:
                speeds[i], speeds[j] = collision(r1, r2, v1, v2, distance) 
                coords[i] += speeds[i]
                coords[j] += speeds[j]

In [20]:
bounds = 0., 10.
part_numb = 50
d = 2.

values = np.linspace(bounds[0], bounds[1]//2-2, int(np.sqrt(part_numb)))

coords = np.zeros((part_numb, 2), dtype=np.float16)
counter = 0
for i in range(int(np.sqrt(part_numb))):
    for j in range(int(np.sqrt(part_numb))):
        coords[counter] = values[i], values[j]
        counter += 1
        
speeds = uniform(-bounds[1], bounds[1], size=(part_numb, 2)) * .015
# speeds = np.ones_like(coords)

%matplotlib qt

GRAY = "#333"
fig, ax = plt.subplots()
fig.set_figwidth(7)
fig.set_figheight(7)
fig.set_facecolor('#222')

for n in range(500):
    colors = np.sqrt(np.square(speeds).sum(axis=1))
    ax.clear()
    ax.scatter(coords[:, 0], coords[:, 1], cmap='hot', c=colors, s=50)
    ax.vlines(bounds[1]//2-1, *bounds)
    ax.vlines(bounds[1]//2+1, *bounds)
    one_step(coords, speeds, bounds)
    ax.set_xlim(bounds)
    ax.set_ylim(bounds)
    ax.tick_params('both', length=0, colors='grey')
    ax.set_facecolor(GRAY)
    ax.grid()
    plt.pause(0.05)

In [21]:
import numpy as np

def wkb_transmission(v, V0=1.0, a=2.0, m=1.0, hbar=1.0):
    E = 0.5 * m * np.dot(v, v)
    if E >= V0:
        # можно взять T≈1, или чуть «рифлёную» формулу; для простоты:
        return 1.0
    kappa = np.sqrt(2.0 * m * max(V0 - E, 0.0)) / hbar
    return float(np.exp(-2.0 * kappa * a))

def reflect_at_plane(x_new, x_plane):
    # зеркалим положение относительно вертикальной границы
    return x_plane - (x_new - x_plane)

def one_step(coords, speeds, bounds, V0=1.0, a=2.0, m=1.0, hbar=1.0):
    # Параметры барьера
    left  = bounds[1] / 2.0 - 1.0
    right = bounds[1] / 2.0 + 1.0
    r = d * 0.5
    dt = 1.0  # при желании вынесите наружу

    old = coords.copy()
    coords += speeds * dt

    # --- внешние стены (с учётом радиуса) ---
    for axis in (0, 1):
        lo = bounds[0] + r
        hi = bounds[1] - r

        hit_lo = coords[:, axis] < lo
        if np.any(hit_lo):
            coords[hit_lo, axis] = lo + (lo - coords[hit_lo, axis])  # зеркалим
            speeds[hit_lo, axis] *= -1.0

        hit_hi = coords[:, axis] > hi
        if np.any(hit_hi):
            coords[hit_hi, axis] = hi - (coords[hit_hi, axis] - hi)  # зеркалим
            speeds[hit_hi, axis] *= -1.0

    # --- внутренняя стенка как две вертикальные линии x=left и x=right ---
    # Пересечение левой границы: шли слева направо и перескочили через x=left
    cross_left = (old[:, 0] + r <= left) & (coords[:, 0] + r > left) & (speeds[:, 0] > 0)
    idxL = np.where(cross_left)[0]
    for i in idxL:
        T = wkb_transmission(speeds[i], V0=V0, a=a, m=m, hbar=hbar)
        if np.random.rand() > T:
            # отражаемся на левой грани
            coords[i, 0] = reflect_at_plane(coords[i, 0], left - r)
            speeds[i, 0] *= -1.0

    # Пересечение правой границы: шли справа налево и перескочили через x=right
    cross_right = (old[:, 0] - r >= right) & (coords[:, 0] - r < right) & (speeds[:, 0] < 0)
    idxR = np.where(cross_right)[0]
    for i in idxR:
        T = wkb_transmission(speeds[i], V0=V0, a=a, m=m, hbar=hbar)
        if np.random.rand() > T:
            # отражаемся на правой грани
            coords[i, 0] = reflect_at_plane(coords[i, 0], right + r)
            speeds[i, 0] *= -1.0

    # --- столкновения шариков (упруго, равные массы) ---
    # быстрый и устойчивый вариант без поворотов базиса
    eps = 1e-9
    N = len(coords)
    for i in range(N - 1):
        for j in range(i + 1, N):
            rij = coords[i] - coords[j]
            dist2 = np.dot(rij, rij)
            min2 = (2.0 * r) ** 2
            if dist2 < min2 and dist2 > eps:
                dist = np.sqrt(dist2)
                n = rij / dist
                # минимальное разведение (position correction)
                overlap = 2.0 * r - dist
                coords[i] += 0.5 * overlap * n
                coords[j] -= 0.5 * overlap * n
                # относительная скорость вдоль нормали
                dv = speeds[i] - speeds[j]
                vn = np.dot(dv, n)
                if vn < 0.0:
                    # обмен импульсом вдоль нормали (равные массы)
                    impulse = vn * n
                    speeds[i] -= impulse
                    speeds[j] += impulse


In [31]:
np.random.seed(0)

bounds = (0.0, 100.0)
part_numb = 50
d = 0.4  # разумный диаметр; r=0.2

# стартуем слева от барьера, с зазором от стен
r = d * 0.5
xmin, xmax = bounds[0] + r, bounds[1] / 2.0 - 1.2 - r  # до левого края барьера минус зазор
ymin, ymax = bounds[0] + r, bounds[1] - r

coords = np.column_stack([
    np.random.uniform(xmin, xmax, part_numb),
    np.random.uniform(ymin, ymax, part_numb),
]).astype(np.float64)

speeds = np.random.uniform(-1.0, 1.0, size=(part_numb, 2)).astype(np.float64) * 0.3


In [32]:
%matplotlib qt

GRAY = "#333"
fig, ax = plt.subplots()
fig.set_figwidth(7)
fig.set_figheight(7)
fig.set_facecolor('#222')

for n in range(500):
    colors = np.sqrt(np.square(speeds).sum(axis=1))
    ax.clear()
    ax.scatter(coords[:, 0], coords[:, 1], cmap='hot', c=colors, s=50)
    ax.vlines(bounds[1]//2-1, *bounds)
    ax.vlines(bounds[1]//2+1, *bounds)
    one_step(coords, speeds, bounds)
    ax.set_xlim(bounds)
    ax.set_ylim(bounds)
    ax.tick_params('both', length=0, colors='grey')
    # ax.axis(False)
    ax.set_facecolor(GRAY)
    ax.grid()
    plt.pause(0.05)

KeyboardInterrupt: 

In [33]:
%matplotlib qt

GRAY = "#333"
fig, ax = plt.subplots()
fig.set_figwidth(7)
fig.set_figheight(7)
fig.set_facecolor('#222')

# границы «барьера» как полоса шириной 2
mid = (bounds[0] + bounds[1]) / 2.0
left, right = mid - 1.0, mid + 1.0

for n in range(500):
    colors = np.linalg.norm(speeds, axis=1)
    ax.clear()

    # белая полоса с прозрачностью 0.5 по центру
    ax.axvspan(left, right, ymin=0, ymax=1, facecolor='white', alpha=0.2, zorder=0)

    ax.scatter(coords[:, 0], coords[:, 1], cmap='hot', c=colors, s=50, zorder=1)

    one_step(coords, speeds, bounds)

    ax.set_xlim(bounds)
    ax.set_ylim(bounds)

    # убрать надписи и деления на осях
    ax.set_xticks([]); ax.set_yticks([])
    ax.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)

    ax.set_facecolor(GRAY)
    ax.grid()
    plt.pause(0.05)
